In [ ]:
import shutil
import pyemu
from pathlib import Path
# from pyemu.utils import VisHandler
from pyemu import vis_utils
from pyemu import os_utils
import sys
import flopy
sys.path.append('..')
from autotest.pst_from_tests import _get_port, ies_exe_path, mf6_exe_path

In [ ]:
m_d = Path("pst_template")
t_d = Path(".", "vis_eg", 'template')

In [ ]:
if t_d.exists():
    shutil.rmtree(t_d)
shutil.copytree(m_d, t_d)

In [ ]:
pst = pyemu.Pst(str(t_d / "freyberg.pst"))
pst.pestpp_options['ies_num_reals'] = 10
pst.write(pst.filename, version=2)
port = _get_port()
m_d = t_d.with_name("master")
shutil.copy(shutil.which(ies_exe_path), t_d)
shutil.copy(shutil.which(mf6_exe_path), t_d)
os_utils.start_workers(t_d,"pestpp-ies","freyberg.pst",num_workers=5,
                             worker_root=t_d.parent,
                             master_dir=m_d, port=port)

In [ ]:
pst = pyemu.Pst(str(m_d / "freyberg.pst"))
obs = pst.observation_data
obs.loc[obs.oname=='hds', ['k', 'i', 'j']] = obs.loc[obs.oname=='hds'].obgnme.str.rsplit("_",expand=True, n=3)[[1,2,3]].values
pst.observation_data = obs
sim = flopy.mf6.MFSimulation.load(sim_ws=m_d, verbosity_level=0)
m = sim.get_model("freyberg6")
mg = m.modelgrid
mg.set_coord_info(xoff=622241.1904510253, yoff=3343617.741737109, angrot=15.0)
m.dis.xorigin = mg.xoffset
m.dis.yorigin = mg.yoffset
m.dis.angrot = mg.angrot
sim.write_simulation()

In [ ]:
vh = vis_utils.VisHandler(pst, wd=m_d, crs="epsg:32614")

In [ ]:
display(vh.default_map_layout)

In [ ]:
vh.default_unmap_layout

In [ ]:
m.modelgrid